# HTML Parser for TME Music Theory Treatises

### Overview
This notebook processes HTML files of English music theory treatises and creates a vector database for semantic search using LangChain and ChromaDB.

### Key Features of This Notebook
This notebook now uses an intelligent incremental update approach:

1. CONFIGURATION TRACKING (db_config.json)
   - Tracks embedding model, chunk size, and other settings
   - Only recreates DB when breaking changes are detected
   - Breaking changes: embedding model or chunk size changes

2. FILE CHANGE DETECTION (file_hashes.json)
   - Tracks MD5 hash of each source XML file
   - Only reprocesses files that have changed
   - Skips unchanged files automatically

3. DOCUMENT ID MANAGEMENT
   - Each chunk gets a unique, deterministic ID
   - Allows updating existing documents without duplicates
   - Format: MD5(source_file_page_number_chunk_index)

4. INCREMENTAL UPDATES
   - When a file changes, old documents are deleted first
   - New documents are added with same IDs if content unchanged
   - Prevents duplicate embeddings

### Common Operations:

```python
# Add or update files
process_xml_files()  # Only processes changed files

# Force complete rebuild (rare)
process_xml_files(force_reprocess=True)

# View database statistics
get_db_stats()

# Remove a specific file
delete_source_file("MARLU9.html")

# Search the database
results = vector_store.similarity_search("your query here", k=5)

# Search with scores
results = vector_store.similarity_search_with_score("your query", k=5)
```


### Main Steps:
1. **Import Libraries** - Load necessary Python packages
2. **Configure Database** - Set up ChromaDB with intelligent update tracking
3. **Parse TEI XML** - Extract metadata and text from treatises
4. **Create Embeddings** - Generate vector embeddings using OpenAI
5. **Store & Query** - Save to ChromaDB and explore the database

---

### Step 1: Import Required Libraries and API Key

In [1]:
# Standard library imports
import glob
import hashlib
import json
import os
import re
import shutil
import getpass
from pathlib import Path

# Third-party imports
import pandas as pd
from bs4 import BeautifulSoup, NavigableString

# LangChain imports
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


def read_file_with_fallback_encoding(filepath):
    """
    Read a file trying multiple encodings.
    Returns the file content as a string.
    
    Tries: utf-8, latin-1 (ISO-8859-1), cp1252 (Windows)

    Args:
        filepath (str or Path): Path to the file to read
    """
    encodings = ['utf-8', 'latin-1', 'cp1252']
    
    for encoding in encodings:
        try:
            with open(filepath, 'r', encoding=encoding) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
    
    # Last resort: read with errors='replace' to substitute problematic chars
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        return f.read()

/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

### Step 2: Configure OpenAI API Key and ChromaDB Settings

We use OpenAI's embedding model to convert text into vectors for semantic search.

In [2]:
# Prompt for OpenAI API key with password masking
print("Please enter your OpenAI API key:")
openai_api_key = getpass.getpass("API Key: ")

# Set as environment variable
os.environ["OPENAI_API_KEY"] = openai_api_key

# Verify it was set (show only first/last few characters for security)
if openai_api_key:
    masked_key = f"{openai_api_key[:7]}...{openai_api_key[-4:]}"
    print(f"✓ API key set successfully: {masked_key}")
else:
    print("✗ No API key entered")



Please enter your OpenAI API key:
✓ API key set successfully: sk-proj...NogA


In [3]:

# Configuration for database schema and settings--these will be passed to all the relevant components below
DB_CONFIG = {
    "version": "1.0",
    "embedding_model": "text-embedding-3-small",
    "chunk_size": 2000,
    "chunk_overlap": 300,
    "collection_name": "tme_english"
}

db_path = Path('./chroma-db_tme_english')
config_path = db_path / 'db_config.json'

# Check if we need to recreate the database
should_recreate = False

if db_path.exists() and config_path.exists():
    # Load existing config
    with open(config_path, 'r') as f:
        existing_config = json.load(f)
    
    # Check for breaking changes
    if (existing_config.get('embedding_model') != DB_CONFIG['embedding_model'] or
        existing_config.get('chunk_size') != DB_CONFIG['chunk_size']):
        print(f"⚠️  Breaking changes detected:")
        print(f"   Old: {existing_config}")
        print(f"   New: {DB_CONFIG}")
        should_recreate = True
    else:
        print(f"✓ Using existing database - configuration unchanged")
        print(f"  Will perform incremental updates only")
elif not db_path.exists():
    print(f"✓ Creating new database at {db_path}")
    should_recreate = True
else:
    print(f"⚠️  Database exists but no config found - will recreate")
    should_recreate = True

# Delete database only if necessary
if should_recreate and db_path.exists():
    shutil.rmtree(db_path)
    print(f"✓ Deleted existing database at {db_path}")

# Create directory if needed
db_path.mkdir(exist_ok=True)

# Save current configuration
with open(config_path, 'w') as f:
    json.dump(DB_CONFIG, f, indent=2)

# Initialize embeddings
embeddings = OpenAIEmbeddings(model=DB_CONFIG['embedding_model'])

# Initialize Chroma vector store
vector_store = Chroma(
    collection_name=DB_CONFIG['collection_name'],
    embedding_function=embeddings,
    persist_directory=str(db_path)
)

# Configure text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=DB_CONFIG['chunk_size'],
    chunk_overlap=DB_CONFIG['chunk_overlap'],
    length_function=len,
    is_separator_regex=False
)

✓ Using existing database - configuration unchanged
  Will perform incremental updates only


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


---

### Step 3: Initialize ChromaDB Vector Store

### What is a Vector Database?
A vector database stores text as numerical vectors (embeddings) that capture semantic meaning. This allows us to find relevant passages based on meaning, not just keyword matching.

### Intelligent Update Strategy
This notebook implements a **smart incremental update system**:
- **Configuration Tracking**: Detects breaking changes (embedding model, chunk size)
- **File Change Detection**: Only reprocesses modified XML files
- **No Unnecessary Rebuilds**: Saves time and API costs

The system creates two tracking files:
- `db_config.json` - Stores database configuration
- `file_hashes.json` - Tracks which files have been processed

In [4]:
# metadata extraction function for the TME English music theory files
# the metadata will be used in the chroma db and also in the csv file for the Streamlit app as a record of our sources

def parse_century_to_years(century_num):
    """
    Convert century number to year range and return these as start and end years in the metadata
    
    Examples:
        15 -> (1400, 1499)
        16 -> (1500, 1599)
        14 -> (1300, 1399)
    
    Returns: (start_year, end_year)    

    Args:
        century_num (int): Century number (e.g., 15 for 15th century
    """
    return ((century_num - 1) * 100, century_num * 100 - 1)


def extract_century_from_html(soup):
    """
    Extract century from <center><a href="15th.html">Return to 15th-Century Filelist</a></center>
    
    Returns: tuple (century_string, date_start, date_end) or 'undetected' if not found.
    century_string is for example "15th century"

    Args:
        soup (BeautifulSoup): Parsed HTML content
    """
    # Find <center> tag containing an <a> tag
    center_tags = soup.find_all('center')
    for center in center_tags:
        a_tag = center.find('a')
        if a_tag:
            text = a_tag.get_text(strip=True)
            # Look for pattern like "15th-Century" or "16th-Century"
            century_match = re.search(r'(\d+)(?:st|nd|rd|th)-Century', text, re.IGNORECASE)
            if century_match:
                century_num = int(century_match.group(1))
                # Build ordinal suffix
                if century_num == 1:
                    suffix = "st"
                elif century_num == 2:
                    suffix = "nd"
                elif century_num == 3:
                    suffix = "rd"
                else:
                    suffix = "th"
                century_string = f"{century_num}{suffix} century"
                date_start, date_end = parse_century_to_years(century_num)
                return (century_string, date_start, date_end)
    return None


def extract_metadata(soup, html_path):
    """Extract metadata from TME HTML file.

    Args:
        soup (BeautifulSoup): Parsed HTML content
        html_path (str or Path): Path to the HTML file  

    Returns:
        dict: Metadata dictionary with keys:
            - Filename
            - Title
            - Author
            - Date  (display string)
            - date_start (int)
            - date_end (int)
            - Citation  
    
    """
    if isinstance(html_path, str):
        html_path = Path(html_path)
    
    # Find the <p> tag containing "Fn and Ft:"
    metadata_p = None
    for p in soup.find_all("p"):
        if "Fn and Ft:" in p.get_text():
            metadata_p = p
            break
    
    if not metadata_p:
        return None
    
    # Defaults
    author = "Unknown Author"
    title = "Unknown Title"
    source = "Unknown Source"
    
    # Parse <br> tags - text follows each <br>
    br_tags = metadata_p.find_all("br")
    for br in br_tags:
        # Peek at first text to determine field type
        first_sibling = br.next_sibling
        if not first_sibling:
            continue
        
        first_text = str(first_sibling).strip() if isinstance(first_sibling, NavigableString) else first_sibling.get_text().strip()
        
        # Determine stop condition based on field type
        # Source continues until <p> tag; Author/Title stop at next <br>
        is_source = first_text.startswith('Source:')
        
        # Collect text until appropriate stop tag
        text_parts = []
        for sibling in br.next_siblings:
            # Stop conditions
            if is_source and sibling.name == 'p':
                break
            if not is_source and sibling.name == 'br':
                break
            
            if isinstance(sibling, NavigableString):
                text_parts.append(str(sibling))
            else:
                text_parts.append(sibling.get_text())
        
        text = ''.join(text_parts).strip()
        
        if text.startswith('Author:'):
            author_raw = text.replace("Author:", "").strip()
            # Convert "Surname, Given Name" to "Given Name Surname"
            if ',' in author_raw:
                parts = [p.strip() for p in author_raw.split(',', 1)]  # Split only on first comma
                author = f"{parts[1]} {parts[0]}" if len(parts) == 2 else author_raw
            else:
                author = author_raw
        elif text.startswith('Title:'):
            title = text.replace("Title:", "").strip()
        elif text.startswith('Source:'):
            source = text.replace("Source:", "").strip().replace('\n', ' ').replace('  ', ' ')

    
    # Extract date from source field
    years = re.findall(r'\b(1[4-6]\d{2})\b', source)  # Match years 1400-1699
    
    if years:
        year = int(years[0])
        # this prevents us from getting dates of modern reprints
        if year < 1800:
            date_display = str(year)
            date_start = date_display[0:2] + "00"   
            date_end = date_display[0:2] + "99"
    else:
        # Fallback: extract century from footer
        century_info = extract_century_from_html(soup)
        if century_info:
            date_display, date_start, date_end = century_info
        else:
            # Default to 16th century range if nothing found
            date_display = "undetected"
            date_start = "undetected"
            date_end = "undetected"
    
    return {
        "filename": html_path.name,
        "title": title,
        "author": author,
        "date": date_display,
        "date_start": date_start,
        "date_end": date_end,
        "citation": source
    }



def extract_pages_and_text(html_content, html_path):
    """
    Extract page numbers and their associated text from HTML.
    
    Page numbers are marked with [-XX-] pattern (e.g., [-68-], [-72-]) in the text.
    
    Args:
        html_content: Either a string containing HTML or a BeautifulSoup object
        html_path: Path to the HTML file    
        
    Returns:
        List of dictionaries with 'PageNumber', 'PageText', and metadata
    """
    if isinstance(html_content, BeautifulSoup):
        soup = html_content
    else:
        soup = BeautifulSoup(html_content, 'html.parser')

    metadata = extract_metadata(soup, html_path)
    
    # Get text from <body> directly to avoid duplication from unclosed <p> tags
    # Old-style HTML often uses <p> as a separator without closing tags,
    # which BeautifulSoup interprets as nested tags, causing massive text duplication
    body = soup.find('body')
    if body:
        full_text = body.get_text(separator='\n')
    else:
        # Fallback: get all text from the document
        full_text = soup.get_text(separator='\n')
    
    # Pattern to match page markers: [-XX-] where XX is alphanumeric
    page_marker_pattern = r'\[-([\w]+)-\]'
    
    # Find all page markers and their positions
    markers = list(re.finditer(page_marker_pattern, full_text))
    
    if not markers:
        return []
    
    results = []
    
    for i, match in enumerate(markers):
        page_number = match.group(1).strip()
        
        # Text starts after this marker
        start_pos = match.end()
        
        # Text ends at the next marker (or end of document)
        if i + 1 < len(markers):
            end_pos = markers[i + 1].start()
        else:
            end_pos = len(full_text)
        
        page_text = full_text[start_pos:end_pos].strip()
        
        result_entry = {
            'pageNumber': page_number,
            'pageText': page_text
        }
        
        if metadata:
            result_entry.update(metadata)
        
        results.append(result_entry)
    
    return results

In [5]:
html_path = Path('./english_sources/TUCKE_MLBLA103.html')
html_content = read_file_with_fallback_encoding(html_path)
soup = BeautifulSoup(html_content, 'html.parser')


# print(source)

---

### Step 4: Define TEI XML Parsing Functions

These functions extract structured data from TEI XML files:

`extract_metadata(soup, html_path)`
Extracts bibliographic information from the TEI header:
- Title of the treatise
- Author name
- Date/century
- Source filename

`extract_pages_and_text(xml_content, xml_path)`
Extracts page-by-page text content:
- Identifies page breaks (`<pb>` tags)
- Collects all text between page breaks
- Preserves document structure
- Returns list of pages with metadata

---

### Step 5: Test Text Extraction (Optional)

**Purpose**: This cell is for testing/debugging only. It extracts text and metadata from XML files without creating embeddings.


**Note**: You can skip this cell and go directly to `process_xml_files()` for normal operation.

In [6]:
# run this on all files in the source directory:  this is just to get the text, not the vector db
html_dir = Path("english_sources")
all_results = {}
for html_path in html_dir.glob("*.html"):
    print(f"Processing {html_path.name}...")
    html_content = read_file_with_fallback_encoding(html_path)
    results = extract_pages_and_text(html_content, html_path)
    all_results[html_path.name] = results

Processing POWERTR3_TEXT.html...
Processing CHORLAM2_TEXT.html...
Processing DEPRPC2_TEXT.html...
Processing CAXMIR1_TEXT.html...
Processing DEPRPA3_TEXT.html...
Processing ITESTO3_TEXT.html...
Processing DEPRPA1A_MNYPMB12.html...
Processing LEAVPHAR_TEXT.html...
Processing ANONOTE2_TEXT.html...
Processing CHORLAM1_MLBLA292.html...
Processing PSAL1562_TEXT.html...
Processing CHORLAM4_TEXT.html...
Processing MOR1597A_TEXT.html...
Processing DEEPRE_TEXT.html...
Processing DEPRPC3_TEXT.html...
Processing POWERTR2_TEXT.html...
Processing CHORLAM3_TEXT.html...
Processing CANTROS_TEXT.html...
Processing DEPRPA2_TEXT.html...
Processing ITESTO2_TEXT.html...
Processing DEPRPB1_MLBLL763.html...
Processing LANTLIT_TEXT.html...
Processing BARTREV1_TEXT.html...
Processing LECPROV1_MLBLR18.html...
Processing DEPRPC1B_MLBLL763.html...
Processing OFPRELA_TEXT.html...
Processing POWERTR4_TEXT.html...
Processing CHORLAM5_TEXT.html...
Processing ANOPRAIM_TEXT.html...
Processing CORNPAR2_MLBLH43.html...
P

In [7]:
# just to check results for ONE file 
all_results["RAVBD_TEXT.html"]

[{'pageNumber': '1',
  'pageText': 'The Definitions and Diuisions of Moode Time, and Prolation in Measurable \nMusick.\n\nMEnsurabilis Musice is defined to be a Harmony of diuers sortes of Sounds, exprest \nby certaine Characters or Figures called Notes, describd on Lines and Spaces, different in \nName, Essence, Forme, Quantity, and Quality, which are sung by a Measure of Time; or \nas (1) Iohn Dunstable [(1) Iohn Dunstable Mensurabilis Musica cap I. in marg.], \n(2) the man whom Ioannes Nucius in his Poeticall Musicke (and diuers others) affirme to \nbe the first that inuented Composition) saith [(2) Iohannes Nucius musica Poatica capitulum \nI. in marg.], it hath his beginning at an Vnite, and increaseth vpward by two and by three \ninfinitely, and from the highest decreaseth in like manner downe againe to an Vnite.\n\nMeasure in this Science is a Quantity of the length and shortnes of Time, either by \nNaturall sounds pronounced by Voice, or by Artificiall, vpon Instruments.\n\nOf 

---
### Get the Metadata DataFrame


In [8]:
# test metaddata df
html_dir = Path("english_sources")
metadata = []
for html_path in html_dir.glob("*.html"):
    html_content = read_file_with_fallback_encoding(html_path)
    one_document_metadata = extract_metadata(BeautifulSoup(html_content, 'html.parser'), html_path)  
    metadata.append(one_document_metadata)   
metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv("english_html_metadata.csv", index=False)
metadata_df

,filename,title,author,date,date_start,date_end,citation
0,POWERTR3_TEXT.html,Treatise upon the Gamme,Leonel Power,15th century,1400,1499,"Manfred Bukofzer, Geschichte des englischen Di..."
1,CHORLAM2_TEXT.html,A Chorister's Lament,Anonymous,14th century,1300,1399,"Moriz Haupt and Heinrich Hoffmann, Altdeutsche..."
2,DEPRPC2_TEXT.html,On the three manners of proportions,"Anonymous (""secundum Chilston"")",15th century,1400,1499,"Sanford B. Meech, ""Three Musical Treatises in ..."
3,CAXMIR1_TEXT.html,Mirror of the World (excerpt),tr. Caxton Gautier (or Gossouin) de Metz,1481,1400,1499,"Oliver H. Prior, ed., Caxton's Mirrour of the ..."
4,DEPRPA3_TEXT.html,On the nature of proportions,Anonymous,15th century,1400,1499,"Sir John Hawkins, A General History of the Sci..."
...,...,...,...,...,...,...,...
65,FEIGNCON_TEXT.html,Of Feigned Contemplative Life (exerpt),John Wycliffe,14th century,1300,1399,"Frederic D. Matthews, ed., The English Works o..."
66,CORNPAR4_TEXT.html,A Treatise between Information and Truth,William Cornysh,16th century,1500,1599,"Sir John Hawkins, A General History of the Sci..."
67,BARTREV5_TEXT.html,On the Properties of Things (excerpt),trans. John Trevisa Bartholomaeus Anglicus,14th century,1300,1399,"Sir John Hawkins, A General History of the Sci..."
68,HEREFOL2_TEXT.html,A little treatise on discant,Anonymous,15th century,1400,1499,"Sanford B. Meech, ""Three Musical Treatises in ..."


## Functions to Process all the Files and Create/Update the ChromaDB

In [9]:

def generate_document_id(source_file, page_range, chunk_index):
    """Generate a unique, deterministic ID for each document chunk.
    
    Args:
        source_file: Source filename or identifier
        page_range: Page range string (e.g., "42" or "42-45")
        chunk_index: Index of the chunk within the document (a hashable number)

    Returns:
        str: A unique, deterministic ID for the document chunk
    """
    id_string = f"{source_file}_{page_range}_chunk_{chunk_index}"
    return hashlib.md5(id_string.encode()).hexdigest()  

def get_file_hash(filepath):
    """Get MD5 hash of a file to detect changes.

    Args:
        filepath: Path to the file to hash

    Returns:
        str: MD5 hash of the file
    """
    hash_md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

def load_file_hashes():
    """Load previously processed file hashes.

    Returns a dictionary mapping filenames to their MD5 hashes."""

    hash_file = db_path / 'file_hashes.json'
    if hash_file.exists():
        with open(hash_file, 'r') as f:
            return json.load(f)
    return {}

def save_file_hashes(hashes):
    """Save file hashes to track what's been processed."""
    hash_file = db_path / 'file_hashes.json'
    with open(hash_file, 'w') as f:
        json.dump(hashes, f, indent=2)


def get_page_range_for_chunk(chunk_start, chunk_end, page_boundaries):
    """
    Determine which page(s) a chunk spans based on character positions.
    
    Args:
        chunk_start: Starting character position of chunk in combined text
        chunk_end: Ending character position of chunk
        page_boundaries: List of (page_number, start_pos, end_pos) tuples
    
    Returns:
        String like "42" for single page or "42-45" for page range
    """
    pages_in_chunk = []
    for page_num, page_start, page_end in page_boundaries:
        # Check if chunk overlaps with this page
        if chunk_start < page_end and chunk_end > page_start:
            pages_in_chunk.append(page_num)
    
    if not pages_in_chunk:
        return "unknown"
    elif len(pages_in_chunk) == 1:
        return pages_in_chunk[0]
    else:
        return f"{pages_in_chunk[0]}-{pages_in_chunk[-1]}"


def process_xml_files(xml_dir='english_sources', force_reprocess=False):
    """
    Process all HTML files in the specified directory.
    
    Combines all pages from each source into a single text before chunking,
    so chunk count is proportional to text volume, not page count.
    Page ranges are preserved in metadata.
    
    Args:
        xml_dir: Directory containing HTML files
        force_reprocess: If True, reprocess all files regardless of changes
    """
    xml_files = glob.glob(os.path.join(xml_dir, '*.html'))
    
    if not xml_files:
        print(f"No HTML files found in {xml_dir}")
        return
    
    # Load existing file hashes to detect changes
    existing_hashes = load_file_hashes()
    new_hashes = {}
    
    total_chunks = 0
    total_pages = 0
    files_processed = 0
    files_skipped = 0
    files_updated = 0
    
    # ChromaDB has a max batch size of 5461, so we batch to stay under that
    BATCH_SIZE = 5000
    
    for filepath in xml_files:
        try:
            filename = os.path.basename(filepath)
            current_hash = get_file_hash(filepath)
            new_hashes[filename] = current_hash
            
            # Skip if file hasn't changed (unless force_reprocess is True)
            if not force_reprocess and filename in existing_hashes:
                if existing_hashes[filename] == current_hash:
                    print(f"⊙ {filename} - No changes, skipping")
                    files_skipped += 1
                    continue
                else:
                    print(f"↻ {filename} - File changed, updating...")
                    files_updated += 1
                    # Delete old documents for this file
                    try:
                        vector_store.delete(where={"filename": filename})
                        print(f"  Deleted old documents for {filename}")
                    except Exception as e:
                        print(f"  Note: Could not delete old documents: {e}")
            else:
                print(f"+ {filename} - New file, processing...")
            
            # Read file with fallback encoding support
            xml_content = read_file_with_fallback_encoding(filepath)
            
            # Parse HTML to return the pages in the source document, with metadata
            pages = extract_pages_and_text(xml_content, Path(filepath))
            
            if not pages:
                print(f"  Warning: No pages found in {filepath}")
                continue
            
            # Combine all pages into one text, tracking page boundaries
            combined_text = ""
            page_boundaries = []  # List of (page_number, start_pos, end_pos)
            
            for page in pages:
                start_pos = len(combined_text)
                page_text = page['pageText'].strip()
                if page_text:  # Only add non-empty pages
                    combined_text += page_text + "\n\n"
                    end_pos = len(combined_text)
                    page_boundaries.append((page['pageNumber'], start_pos, end_pos))
            
            if not combined_text.strip():
                print(f"  Warning: No text content in {filepath}")
                continue
            
            # Get document-level metadata from first page
            doc_metadata = {
                "title": pages[0].get('title', 'Unknown'),
                "author": pages[0].get('author', 'Unknown'),
                "date": pages[0].get('date', 'Unknown'),
                "date_start": pages[0].get('date_start', 1500),
                "date_end": pages[0].get('date_end', 1599),
                "citation": pages[0].get('citation', 'Unknown'),
                "filename": filename
            }
            
            # Chunk the combined text
            chunks = text_splitter.create_documents(
                texts=[combined_text],
                metadatas=[doc_metadata]
            )
            
            # Now determine page range for each chunk and update metadata
            all_chunks = []
            all_chunk_ids = []
            current_pos = 0
            
            for i, chunk in enumerate(chunks):
                # Find where this chunk is in the combined text
                chunk_text = chunk.page_content
                chunk_start = combined_text.find(chunk_text, current_pos)
                if chunk_start == -1:
                    chunk_start = current_pos  # Fallback
                chunk_end = chunk_start + len(chunk_text)
                current_pos = chunk_start + 1  # Move past for next search
                
                # Determine page range
                page_range = get_page_range_for_chunk(chunk_start, chunk_end, page_boundaries)
                
                # Update chunk metadata with page range
                chunk.metadata['page_range'] = page_range
                
                # Generate unique ID
                chunk_id = generate_document_id(
                    doc_metadata['citation'],
                    page_range,
                    i
                )
                
                all_chunks.append(chunk)
                all_chunk_ids.append(chunk_id)
            
            # Add all chunks for this file in batches (ChromaDB max batch size is 5461)
            if all_chunks:
                for batch_start in range(0, len(all_chunks), BATCH_SIZE):
                    batch_end = min(batch_start + BATCH_SIZE, len(all_chunks))
                    batch_chunks = all_chunks[batch_start:batch_end]
                    batch_ids = all_chunk_ids[batch_start:batch_end]
                    vector_store.add_documents(
                        documents=batch_chunks,
                        ids=batch_ids
                    )
                    if len(all_chunks) > BATCH_SIZE:
                        print(f"    Added batch {batch_start//BATCH_SIZE + 1}: chunks {batch_start+1}-{batch_end}")
            
            total_chunks += len(all_chunks)
            total_pages += len(pages)
            files_processed += 1
            
            # Show first and last page numbers
            first_page = page_boundaries[0][0] if page_boundaries else "?"
            last_page = page_boundaries[-1][0] if page_boundaries else "?"
            
            print(f'  ✓ Title: {doc_metadata["title"][:80]}...' if len(doc_metadata["title"]) > 80 else f'  ✓ Title: {doc_metadata["title"]}')
            print(f'    Author: {doc_metadata["author"]} | Date: {doc_metadata["date_start"]}-{doc_metadata["date_end"]}')
            print(f'    Citation: {doc_metadata["citation"][:50]}...' if len(doc_metadata["citation"]) > 50 else f'    Citation: {doc_metadata["citation"]}')
            print(f'    Pages: {len(pages)} (pp. {first_page}-{last_page}) | Chunks: {len(all_chunks)}')
            
        except Exception as e:
            print(f"✗ Error processing {filepath}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    # Save the new hashes
    save_file_hashes(new_hashes)
    
    print(f'\n{"="*50}')
    print(f'ChromaDB Processing Complete')
    print(f'{"="*50}')
    print(f'Total files: {len(xml_files)}')
    print(f'  New/Updated: {files_processed}')
    print(f'  Skipped (unchanged): {files_skipped}')
    print(f'  Changed: {files_updated}')
    print(f'Total pages processed: {total_pages}')
    print(f'Total chunks added: {total_chunks}')

---

## Step 6: Process XML Files and Create Vector Database

### What Happens Here:

1. **File Hash Tracking**: Computes MD5 hash of each XML file to detect changes
2. **Smart Processing**: 
   - ⊙ **Skips** unchanged files
   - ↻ **Updates** modified files (deletes old, adds new)
   - \+ **Processes** new files
3. **Text Chunking**: Splits long pages into 2000-character chunks with 300-char overlap
4. **ID Generation**: Creates deterministic IDs for each chunk (prevents duplicates)
5. **Embedding Creation**: Sends chunks to OpenAI for vector embedding
6. **Database Storage**: Stores embeddings and metadata in ChromaDB

### Understanding Chunks vs Pages:
- **Page**: A logical division from the original document (marked by `<pb>` tags)
- **Chunk**: A piece of text ≤2000 characters for optimal embedding
- One page may create multiple chunks if text is long

In [10]:
# Process XML files - only processes new or changed files by default
# Use force_reprocess=True to reprocess everything
process_xml_files(xml_dir='english_sources', force_reprocess=True)

+ POWERTR3_TEXT.html - New file, processing...
  ✓ Title: Treatise upon the Gamme
    Author: Leonel Power | Date: 1400-1499
    Citation: Manfred Bukofzer, Geschichte des englischen Diskan...
    Pages: 5 (pp. 132-136) | Chunks: 5
+ CHORLAM2_TEXT.html - New file, processing...
  ✓ Title: A Chorister's Lament
    Author: Anonymous | Date: 1300-1399
    Citation: Moriz Haupt and Heinrich Hoffmann, Altdeutsche Blä...
    Pages: 2 (pp. 145-146) | Chunks: 2
+ DEPRPC2_TEXT.html - New file, processing...
  ✓ Title: On the three manners of proportions
    Author: Anonymous ("secundum Chilston") | Date: 1400-1499
    Citation: Sanford B. Meech, "Three Musical Treatises in Engl...
    Pages: 2 (pp. 268-269) | Chunks: 6
+ CAXMIR1_TEXT.html - New file, processing...
  ✓ Title: Mirror of the World (excerpt)
    Author: tr. Caxton Gautier (or Gossouin) de Metz | Date: 1400-1499
    Citation: Oliver H. Prior, ed., Caxton's Mirrour of the Worl...
    Pages: 3 (pp. 38-40) | Chunks: 2
+ DEPRPA3_TEXT.ht

In [11]:
import numpy as np

# Get all documents from the database
all_docs = vector_store.get(include=['documents'])

chunk_sizes = [len(doc) for doc in all_docs['documents']]

print(f"Chunk Size Distribution:")
print(f"  25th percentile: {int(np.percentile(chunk_sizes, 25))} chars")
print(f"  50th percentile: {int(np.percentile(chunk_sizes, 50))} chars")
print(f"  75th percentile: {int(np.percentile(chunk_sizes, 75))} chars")
print(f"  90th percentile: {int(np.percentile(chunk_sizes, 90))} chars")



Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Chunk Size Distribution:
  25th percentile: 1283 chars
  50th percentile: 1733 chars
  75th percentile: 1943 chars
  90th percentile: 1982 chars


In [12]:
# Helper functions for database management

def get_db_stats():
    """Get statistics about the current database."""
    all_docs = vector_store.get()
    
    if not all_docs or 'metadatas' not in all_docs:
        print("Database is empty")
        return
    
    total_docs = len(all_docs['ids'])
    
    # Collect statistics
    authors = set()
    citations = set()
    dates = set()
    
    for metadata in all_docs['metadatas']:
        if metadata:
            if 'author' in metadata:
                authors.add(metadata['author'])
            if 'citation' in metadata:
                citations.add(metadata['citation'])
            if 'date' in metadata:
                dates.add(metadata['date'])
    
    print(f"Database Statistics:")
    print(f"  Total chunks: {total_docs}")
    print(f"  Unique authors: {len(authors)}")
    print(f"  Unique citations: {len(citations)}")
    print(f"  Date range: {sorted(dates)}")
    print(f"\nAuthors: {sorted(authors)}")
    
    return {
        'total_docs': total_docs,
        'authors': sorted(authors),
        'citations': sorted(citations),
        'dates': sorted(dates)
    }

def delete_source_file(source_filename):
    """Delete all documents from a specific source file."""
    source_id = f"chtml{source_filename}"
    try:
        vector_store.delete(where={"source": source_id})
        print(f"✓ Deleted all documents from {source_filename}")
        
        # Also remove from file hashes
        hashes = load_file_hashes()
        if source_filename in hashes:
            del hashes[source_filename]
            save_file_hashes(hashes)
            print(f"✓ Removed {source_filename} from tracking")
    except Exception as e:
        print(f"✗ Error deleting documents: {e}")

# Get current database statistics
get_db_stats()

Database Statistics:
  Total chunks: 573
  Unique authors: 13
  Unique citations: 43
  Date range: ['1480', '1481', '14th century', '1586', '1597', '15th century', '1614', '16th century']

Authors: ['Anonymous', 'Anonymous ("secundum Chilston")', 'Anonymous [John Case?]', 'John Skelton', 'John Wycliffe', 'Leonel Power', 'Richard Cutell', 'Thomas Morley', 'Thomas Ravenscroft', 'William Bathe', 'William Cornysh', 'tr. Caxton Gautier (or Gossouin) de Metz', 'trans. John Trevisa Bartholomaeus Anglicus']


{'total_docs': 573,
 'authors': ['Anonymous',
  'Anonymous ("secundum Chilston")',
  'Anonymous [John Case?]',
  'John Skelton',
  'John Wycliffe',
  'Leonel Power',
  'Richard Cutell',
  'Thomas Morley',
  'Thomas Ravenscroft',
  'William Bathe',
  'William Cornysh',
  'tr. Caxton Gautier (or Gossouin) de Metz',
  'trans. John Trevisa Bartholomaeus Anglicus'],
 'citations': ['Andrew Wathey, "Notes on Discant and Mensuration from Fifteenth-Century Oxford," in Trent\'anni di ricerca musicologica: studi in onore di F. Alberto Gallo, ed. Patrizia Dalla Vecchia and Donatella Restani (Rome: Torre d\'Orfeo, 1996), 63-72 at 69-71. Graphics: ANONOTE2 01GF Ed. from: Oxford, Bodleian Library, Lat. misc. d. 83, ff. 63-64.',
  'Anonymous [John Case?], The Praise of Musicke (Oxford: Joseph Barnes, 1586; reprint ed., Hildesheim: Olms, 1980) [STC 4757].',
  'Bruce Holsinger, "Langland\'s Musical Reader: Liturgy, Law, and the Constraints of Performance," Studies in the Age of Chaucer 21 (1999): 99-141

---

## Step 7: Explore Database Contents

These cells demonstrate what's stored in the database and how to access it.

### 7.1: View Sample Metadata

Each chunk in the database has metadata that describes its source.

In [27]:
# Get a sample of documents from the database
sample_docs = vector_store.get(limit=3)

# Display metadata from first 3 chunks
print("=" * 60)
print("SAMPLE METADATA FROM DATABASE")
print("=" * 60)

for i, metadata in enumerate(sample_docs['metadatas'][:3], 1):
    print(f"\n📄 Chunk {i}:")
    print(f"   Title: {metadata.get('title', 'N/A')}")
    print(f"   Author: {metadata.get('author', 'N/A')}")
    print(f"   Date: {metadata.get('date', 'N/A')}")
    print(f"    Start Date: {metadata.get('date_start', 'N/A')}")
    print(f"    End Date: {metadata.get('date_end', 'N/A')}")
    print(f"   Page Range: {metadata.get('page_range', 'N/A')}")
    print(f"   Source File: {metadata.get('filename', 'N/A')}")

SAMPLE METADATA FROM DATABASE

📄 Chunk 1:
   Title: A Briefe Discourse
   Author: Thomas Ravenscroft
   Date: 1614
    Start Date: 1600
    End Date: 1699
   Page Range: 1-2
   Source File: RAVBD_TEXT.html

📄 Chunk 2:
   Title: A Briefe Discourse
   Author: Thomas Ravenscroft
   Date: 1614
    Start Date: 1600
    End Date: 1699
   Page Range: 2
   Source File: RAVBD_TEXT.html

📄 Chunk 3:
   Title: A Briefe Discourse
   Author: Thomas Ravenscroft
   Date: 1614
    Start Date: 1600
    End Date: 1699
   Page Range: 3
   Source File: RAVBD_TEXT.html


### 7.2: View Sample Text Chunks

See what the actual text chunks look like.

In [28]:
# Display text content from sample chunks
print("=" * 60)
print("SAMPLE TEXT CHUNKS")
print("=" * 60)

for i, (doc_text, metadata) in enumerate(zip(sample_docs['documents'][:3], sample_docs['metadatas'][:3]), 1):
    print(f"\n📝 Chunk {i}:")
    print(f"   Source: {metadata.get('title', 'N/A')} (Pages {metadata.get('page_range', 'N/A')})")
    print(f".  Citation: {metadata.get('citation', 'N/A')}")
    print(f"   Length: {len(doc_text)} characters")
    print(f"   Text Preview:")
    print(f"   {'-' * 55}")
    # Show first 300 characters
    preview = doc_text[:300] + "..." if len(doc_text) > 300 else doc_text
    print(f"   {preview}")
    print()

SAMPLE TEXT CHUNKS

📝 Chunk 1:
   Source: A Briefe Discourse (Pages 1-2)
.  Citation: Thomas Ravenscroft, A BRIEFE DISCOVRSE Of the true (but neglected) vse of Charactering the Degrees by their Perfection, Imperfection, and Diminution in Measurable Musicke, against the Common Practise and Custome of these Times (London: Edward Allde for Thomas Adams, 1614; reprint ed., with an introduction by Ian Payne, Clarabricken, Kilkenny, Ireland: Boethius Press, 1984) [STC 20756]. Graphics: RAVBD 01GF-RAVBD 59GF
   Length: 1730 characters
   Text Preview:
   -------------------------------------------------------
   The Definitions and Diuisions of Moode Time, and Prolation in Measurable 
Musick.

MEnsurabilis Musice is defined to be a Harmony of diuers sortes of Sounds, exprest 
by certaine Characters or Figures called Notes, describd on Lines and Spaces, different in 
Name, Essence, Forme, Quantity, and Quali...


📝 Chunk 2:
   Source: A Briefe Discourse (Pages 2)
.  Citation: Thomas Ravenscrof

### 7.3: View Sample Vectors (Embeddings)

**What are vectors?** Numerical representations of text meaning. OpenAI's `text-embedding-3-large` creates 3072-dimensional vectors.

**Why vectors?** They enable semantic search - finding text with similar *meaning*, not just matching keywords.

In [29]:
# Display information about the embedding vectors
import numpy as np

if 'embeddings' in sample_docs and sample_docs['embeddings']:
    print("=" * 60)
    print("SAMPLE EMBEDDING VECTORS")
    print("=" * 60)
    
    for i, (embedding, metadata) in enumerate(zip(sample_docs['embeddings'][:3], sample_docs['metadatas'][:3]), 1):
        print(f"\n🔢 Chunk {i} Vector:")
        print(f"   Source: {metadata.get('title', 'N/A')} (Page {metadata.get('page_number', 'N/A')})")
        print(f"   Vector Dimensions: {len(embedding)}")
        print(f"   Vector Type: {type(embedding)}")
        print(f"   First 10 values: {embedding[:10]}")
        print(f"   Vector stats:")
        print(f"      - Min value: {min(embedding):.6f}")
        print(f"      - Max value: {max(embedding):.6f}")
        print(f"      - Mean value: {np.mean(embedding):.6f}")
        print(f"      - Std deviation: {np.std(embedding):.6f}")
else:
    print("Note: Embeddings not included in sample. Use include=['embeddings'] when calling get().")
    print("\nTo retrieve with embeddings:")
    print("sample_with_embeddings = vector_store.get(limit=3, include=['embeddings', 'documents', 'metadatas'])")

Note: Embeddings not included in sample. Use include=['embeddings'] when calling get().

To retrieve with embeddings:
sample_with_embeddings = vector_store.get(limit=3, include=['embeddings', 'documents', 'metadatas'])


## One Sample Document from the Vector Store

In [30]:
sample_with_embeddings = vector_store.get(limit=1, include=['embeddings', 'documents', 'metadatas'])

# how many dimensions in the embedding vector?
len(sample_with_embeddings['embeddings'][0])

# sample
sample_with_embeddings

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


{'ids': ['423011ad3600c454996808c76faf66c3'],
 'embeddings': array([[ 0.05580744,  0.00877744,  0.03134895, ..., -0.00720605,
          0.01959309, -0.01049348]]),
 'documents': ['The Definitions and Diuisions of Moode Time, and Prolation in Measurable \nMusick.\n\nMEnsurabilis Musice is defined to be a Harmony of diuers sortes of Sounds, exprest \nby certaine Characters or Figures called Notes, describd on Lines and Spaces, different in \nName, Essence, Forme, Quantity, and Quality, which are sung by a Measure of Time; or \nas (1) Iohn Dunstable [(1) Iohn Dunstable Mensurabilis Musica cap I. in marg.], \n(2) the man whom Ioannes Nucius in his Poeticall Musicke (and diuers others) affirme to \nbe the first that inuented Composition) saith [(2) Iohannes Nucius musica Poatica capitulum \nI. in marg.], it hath his beginning at an Vnite, and increaseth vpward by two and by three \ninfinitely, and from the highest decreaseth in like manner downe againe to an Vnite.\n\nMeasure in this Scienc

### 7.4: Perform a Semantic Search

Demonstrate how to search the database by meaning.

In [23]:
# Example: Search for passages about musical intervals
query = "diapente and diatessaron intervals"

print("=" * 60)
print(f"SEMANTIC SEARCH EXAMPLE")
print("=" * 60)
print(f"\nQuery: '{query}'")
print("\nTop 3 most relevant passages:\n")

# Perform similarity search
results = vector_store.similarity_search(query, k=10)

for i, doc in enumerate(results, 1):
    print(f"{'='*60}")
    print(f"Result {i}:")
    print(f"   Title: {doc.metadata.get('title', 'N/A')}")
    print(f"   Author: {doc.metadata.get('author', 'N/A')}")
    print(f"   Page: {doc.metadata.get('page_number', 'N/A')}")
    print(f"   Date: {doc.metadata.get('date', 'N/A')}")
    print(f"\n   Text excerpt:")
    print(f"   {'-'*55}")
    # Show first 400 characters
    preview = doc.page_content[:400] + "..." if len(doc.page_content) > 400 else doc.page_content
    print(f"   {preview}")
    print()

SEMANTIC SEARCH EXAMPLE

Query: 'diapente and diatessaron intervals'

Top 3 most relevant passages:



Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Result 1:
   Title: On the denominations of proportions
   Author: Anonymous
   Page: N/A
   Date: 15th-century

   Text excerpt:
   -------------------------------------------------------
   [Anonymous, Proportions II, 268; text: 12.2, sexdupla, 4, 4.3, 
sesquetercia tercias, 5.3, superbiparciens [terci]as, 6.3, dupla, 7.3, dupla 
sesque-tercia, 8.3, dupla superbiparciens tercias, 9.3, tripla, 10.3, tripla 
sesque-tercia, 6, 6.5, sesque-quinta, 7.5, superbiparciens quintas, 8, 8.5, 
supertriparciens quintas, 9.5, super-quatriparciens quintas, 10.5, dupla, 
11.5, dupla sesque-quinta, 1...

Result 2:
   Title: On the denominations of proportions
   Author: Anonymous
   Page: N/A
   Date: 15th century

   Text excerpt:
   -------------------------------------------------------
   [Anonymous, Proportions II, 268; text: 12.2, sexdupla, 4, 4.3, 
sesquetercia tercias, 5.3, superbiparciens [terci]as, 6.3, dupla, 7.3, dupla 
sesque-tercia, 8.3, dupla superbiparciens tercias, 9.3, tripla, 10.3,

### 7.5: Search with Similarity Scores

See the actual similarity scores to understand how close the matches are.

In [26]:
# Search with similarity scores (lower distance = more similar)
query = "What is beautify in music?"

print("=" * 60)
print(f"SEMANTIC SEARCH WITH SCORES")
print("=" * 60)
print(f"\nQuery: '{query}'")
print("\nResults ranked by similarity:\n")

# Perform similarity search with scores
results_with_scores = vector_store.similarity_search_with_score(query, k=10)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"{'='*60}")
    print(f"Result {i} - Similarity Score: {score:.4f}")
    print(f"   Title: {doc.metadata.get('title', 'N/A')}")
    print(f"   Author: {doc.metadata.get('author', 'N/A')}")
    print(f"   Page: {doc.metadata.get('page_number', 'N/A')}")
    print(f"\n   Text excerpt (first 250 chars):")
    print(f"   {'-'*55}")
    preview = doc.page_content[:250] + "..." if len(doc.page_content) > 250 else doc.page_content
    print(f"   {preview}")
    print()

print("\n💡 Note: Lower scores indicate higher similarity (distance metric)")
print("   Typical range: 0.0 (identical) to 2.0 (very different)")

SEMANTIC SEARCH WITH SCORES

Query: 'What is beautify in music?'

Results ranked by similarity:

Result 1 - Similarity Score: 1.1058
   Title: The Praise of Musicke
   Author: Anonymous [John Case?]
   Page: N/A

   Text excerpt (first 250 chars):
   -------------------------------------------------------
   together as be the stringes et cetera. 
To the same purpose speaketh Athanasius at large in the same place, and his meaning is as 
well to shewe how good and comely an ornament Musicke is in the churche, (which as in 
those daies it was not doubted o...

Result 2 - Similarity Score: 1.1161
   Title: On the Properties of Things (excerpt)
   Author: Bartholomaeus Anglicus, trans. John Trevisa
   Page: N/A

   Text excerpt (first 250 chars):
   -------------------------------------------------------
   De Musica.

As arte of nombres and mesures seruyth to diuinite, so doth the arte of melody 
for musyk; by the whyche accorde and melody is knowe in sowne, and in songe is 
nedeful to kn

---

## Summary and Next Steps

### What You've Built:
✅ A vector database of Latin music theory treatises  
✅ Semantic search capability (search by meaning, not keywords)  
✅ Intelligent incremental update system (saves time and costs)  
✅ Complete metadata tracking (author, title, date, page numbers)  

### Key Concepts:
- **TEI XML**: Text Encoding Initiative format for digital scholarly texts
- **Embeddings**: Numerical representations of text meaning (3072-dimensional vectors)
- **Chunks**: Text pieces ≤2000 characters for optimal embedding
- **Vector Database**: Stores embeddings for fast similarity search
- **Semantic Search**: Finding relevant passages by meaning, not just keywords


vector_store

In [44]:
# one document in the vector store

vector_store.get(limit=1, include=['metadatas', 'documents'])

{'ids': ['d2e3e599f078c8edde75e140094aedfe'],
 'embeddings': None,
 'documents': ['This Tretis is continuid upon the Gamme for hem that wil be \nsingers or makers or techers. For the ferst thing of alle they must know \nhow many cordis of discant ther be.\n\nAs olde men seyen and as men sing now adayes ther be 9. But \nwhoso wil sing manerli and musikli he may not lepe to the 15 in no maner \nof discant. for it longith for no mannys voys, and so ther be but 8 acordis \naftir the discant now usid. And whoso ever wil be a maker he may use no \nmo[re] than 8. And so ther be but 8 fro unisoun unto the 13. But for the \nQuatrebil syghte ther be 9 acordis of discant: the unisoun, 3, 5, 6, 8, 10, 12, \n13 and 15. Of the whech 9 acordis 5 be perfite and 4 imperfite. the 5 \nperfite be the unisoun, 5, 8, 12 and 15. the 4 imperfite be the 3, 6, 10 and \n13. Also thou maist ascende and descende with al maner of cordis excepte \n2 acordis perfite of one kinde as: 2 unisouns, 2 fifts, 2 eyghts, 2 t

In [17]:
# check compatibility of two dbs

# Load both databases
vector_store_english = Chroma(
    persist_directory='./chroma-db_tme_english',
    embedding_function=embeddings
)

vector_store_italian = Chroma(
    persist_directory='./chroma-db_italian',
    embedding_function=embeddings
)

# Get samples from both
sample_english = vector_store_english.get(limit=3)
sample_italian = vector_store_italian.get(limit=3)

print("=" * 70)
print("ENGLISH DATABASE")
print("=" * 70)
print(f"Document count in sample: {len(sample_english['ids'])}")

if sample_english['metadatas']:
    print("\n📄 Sample metadata:")
    for key in sample_english['metadatas'][0].keys():
        print(f"  - {key}")

print("\n" + "=" * 70)
print("ITALIAN DATABASE")
print("=" * 70)
print(f"Document count in sample: {len(sample_italian['ids'])}")

if sample_italian['metadatas']:
    print("\n📄 Sample metadata:")
    for key in sample_italian['metadatas'][0].keys():
        print(f"  - {key}")

# COMPATIBILITY CHECK
print("\n" + "=" * 70)
print("COMPATIBILITY CHECK")
print("=" * 70)

# Compare metadata fields
eng_keys = set(sample_english['metadatas'][0].keys()) if sample_english['metadatas'] else set()
ita_keys = set(sample_italian['metadatas'][0].keys()) if sample_italian['metadatas'] else set()

print(f"\nEnglish fields: {sorted(eng_keys)}")
print(f"Italian fields: {sorted(ita_keys)}")

common = eng_keys & ita_keys
only_eng = eng_keys - ita_keys
only_ita = ita_keys - eng_keys

print(f"\n✓ Common fields: {sorted(common)}")
if only_eng:
    print(f"⚠ Only in English: {sorted(only_eng)}")
if only_ita:
    print(f"⚠ Only in Italian: {sorted(only_ita)}")

# Compare embedding dimensions
if sample_english.get('embeddings') and sample_italian.get('embeddings'):
    dim_eng = len(sample_english['embeddings'][0])
    dim_ita = len(sample_italian['embeddings'][0])
    
    print(f"\nEmbedding dimensions:")
    print(f"  English: {dim_eng}")
    print(f"  Italian: {dim_ita}")
    print(f"  {'✓ COMPATIBLE' if dim_eng == dim_ita else '✗ INCOMPATIBLE'}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


ENGLISH DATABASE
Document count in sample: 0

ITALIAN DATABASE
Document count in sample: 0

COMPATIBILITY CHECK

English fields: []
Italian fields: []

✓ Common fields: []


In [18]:
sample_english

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}